In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix

train = pd.read_csv('../data/processed/train_processed.csv')
test = pd.read_csv('../data/processed/test_processed.csv')

print("Data loaded.")
print(train.shape)

Data loaded.
(1200, 27)


In [2]:
# --- Shared setup ---
le = LabelEncoder()
y = le.fit_transform(train['emotional_state'])

xgb_params = {
    'n_estimators': 200,
    'max_depth': 4,
    'learning_rate': 0.05,
    'subsample': 0.7,
    'colsample_bytree': 0.7,
    'random_state': 42,
    'eval_metric': 'mlogloss',
    'verbosity': 0
}

# --- Model 1: Text Only ---
print("Running Text Only model...")
tfidf = TfidfVectorizer(max_features=300, ngram_range=(1,2), min_df=2, sublinear_tf=True)
X_text = tfidf.fit_transform(train['cleaned_text'])

model_text = XGBClassifier(**xgb_params)
scores_text = cross_val_score(model_text, X_text, y, cv=5, scoring='accuracy')
print(f"Text Only       → Accuracy: {scores_text.mean():.4f} (+/- {scores_text.std():.4f})")

# --- Model 2: Metadata Only ---
print("Running Metadata Only model...")
METADATA_COLS = ['sleep_hours', 'energy_level', 'stress_level', 'duration_min',
                 'time_of_day_enc', 'reflection_quality_enc', 'previous_day_mood_enc']
ambience_cols = [c for c in train.columns if c.startswith('ambience_type_')]
face_cols = [c for c in train.columns if c.startswith('face_emotion_hint_')]
all_meta = METADATA_COLS + ambience_cols + face_cols

X_meta = csr_matrix(train[all_meta].values.astype(float))

model_meta = XGBClassifier(**xgb_params)
scores_meta = cross_val_score(model_meta, X_meta, y, cv=5, scoring='accuracy')
print(f"Metadata Only   → Accuracy: {scores_meta.mean():.4f} (+/- {scores_meta.std():.4f})")

# --- Model 3: Text + Metadata (combined) ---
print("Running Text + Metadata model...")
X_combined = hstack([X_text, csr_matrix(train[all_meta].values.astype(float))])

model_combined = XGBClassifier(**xgb_params)
scores_combined = cross_val_score(model_combined, X_combined, y, cv=5, scoring='accuracy')
print(f"Text + Metadata → Accuracy: {scores_combined.mean():.4f} (+/- {scores_combined.std():.4f})")

Running Text Only model...
Text Only       → Accuracy: 0.5200 (+/- 0.2334)
Running Metadata Only model...
Metadata Only   → Accuracy: 0.1783 (+/- 0.0180)
Running Text + Metadata model...
Text + Metadata → Accuracy: 0.4967 (+/- 0.2409)


In [3]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

models = ['Text Only', 'Metadata Only', 'Text + Metadata']
means = [scores_text.mean(), scores_meta.mean(), scores_combined.mean()]
stds = [scores_text.std(), scores_meta.std(), scores_combined.std()]

plt.figure(figsize=(8, 5))
bars = plt.bar(models, means, yerr=stds, capsize=8,
               color=['#4C72B0', '#DD8452', '#55A868'],
               edgecolor='black', alpha=0.85)

plt.axhline(y=1/6, color='red', linestyle='--', label='Random Baseline (16.7%)')
plt.ylabel('Cross-val Accuracy')
plt.title('Ablation Study — Text vs Metadata vs Combined')
plt.ylim(0, 0.8)
plt.legend()

for bar, mean in zip(bars, means):
    plt.text(bar.get_x() + bar.get_width()/2, mean + 0.02,
             f'{mean:.3f}', ha='center', fontsize=11)

plt.tight_layout()
plt.savefig('../outputs/ablation_study.png', dpi=150)
plt.close()
print("Plot saved to outputs/ablation_study.png")

Plot saved to outputs/ablation_study.png
